# Day 6 — Multi-provider eval runner

Run the same prompt through OpenAI, Anthropic, and a local HF model. Cache results. Compute simple metrics. This is the skeleton your Week 2 large-scale sweep will reuse.

Companion code: [`../src/reliability_maps/`](../src/reliability_maps/).

In [ ]:
import os, sys, asyncio, json, time, hashlib
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic

openai_client = AsyncOpenAI()
anthropic_client = AsyncAnthropic()

CACHE_DIR = Path(".cache/eval_runner")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def cache_key(*parts) -> Path:
    h = hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE_DIR / f"{h}.json"

def cached(fn):
    async def wrapper(prompt, model, **kw):
        k = cache_key(prompt, model, kw)
        if k.exists(): return json.loads(k.read_text())
        out = await fn(prompt, model, **kw)
        k.write_text(json.dumps(out))
        return out
    return wrapper

In [ ]:
@cached
async def openai_complete(prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0):
    t0 = time.time()
    r = await openai_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "text": r.choices[0].message.content,
        "provider": "openai", "model": model,
        "prompt_tokens": r.usage.prompt_tokens,
        "completion_tokens": r.usage.completion_tokens,
        "latency_s": time.time() - t0,
    }

@cached
async def anthropic_complete(prompt: str, model: str = "claude-haiku-4-5", temperature: float = 0.0):
    t0 = time.time()
    r = await anthropic_client.messages.create(
        model=model, temperature=temperature, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "text": r.content[0].text,
        "provider": "anthropic", "model": model,
        "prompt_tokens": r.usage.input_tokens,
        "completion_tokens": r.usage.output_tokens,
        "latency_s": time.time() - t0,
    }

In [ ]:
# Tiny eval set — replace with HotpotQA / GPQA subset later
EVAL_SET = [
    {"qid": "q1", "q": "What is the capital of France?", "gold": "Paris"},
    {"qid": "q2", "q": "In what year did WWII end?", "gold": "1945"},
    {"qid": "q3", "q": "Who wrote Pride and Prejudice?", "gold": "Jane Austen"},
]

MODELS = [
    (openai_complete,    "gpt-4o-mini"),
    (anthropic_complete, "claude-haiku-4-5"),
]

async def run_all():
    rows = []
    tasks = []
    for item in EVAL_SET:
        for fn, model in MODELS:
            async def _go(item=item, fn=fn, model=model):
                r = await fn(item["q"], model=model)
                return {"qid": item["qid"], "q": item["q"], "gold": item["gold"], **r}
            tasks.append(_go())
    return await asyncio.gather(*tasks)

results = await run_all()
df = pd.DataFrame(results)
df

In [ ]:
# Simple exact-match-ish accuracy
import re
def normalize(s): return re.sub(r"[^a-z0-9]", "", s.lower())
df["correct"] = df.apply(lambda r: normalize(r["gold"]) in normalize(r["text"]), axis=1)
df.groupby("model")["correct"].mean()

## What to do next

1. Swap `EVAL_SET` for a 100-question subset of HotpotQA.
2. Add a local provider (load Qwen2-0.5B from notebook 01).
3. Wire in the distractor generator from `src/reliability_maps/distractors.py` so you can sweep over perturbations.
4. Plot accuracy vs perturbation parameter, faceted by model. **This is your first phase-diagram axis.**